In this notebook we'll explore different document loaders (for parsing) and splitters (for chunking) available through `langchain`.

In [ ]:
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import UnstructuredPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from rag_pipeline.parse_documents import parse_document
from langchain_core.documents import Document

from docling.document_converter import DocumentConverter
from docling_core.types.doc.labels import DocItemLabel

In [ ]:
# doc1 contains 4 columns per page, so I'm testing whether the loader can handle this layout

doc1_path = "/Users/ashapatel/Documents/projects/rag_cc/leaflets_and_guidelines/PILs/motion_sickness_PIL.pdf"

# doc 2 is a more standard layout, with 1 column per page

doc2_path = "/Users/ashapatel/Documents/projects/rag_cc/leaflets_and_guidelines/PILs/antacid_PIL.pdf"

## PyPDF loader

This is suited to simple, clean PDFs.

In [ ]:
loader1 = PyPDFLoader(doc1_path)
document1 = loader1.load()

In [ ]:
loader2 = PyPDFLoader(doc2_path)
document2 = loader2.load()

### Explore the extracted page content

In [ ]:
for page_num, page in enumerate(document1):
    print(f"=== Page {page_num + 1} ===")
    print(page.page_content)
    print()

This document loader correctly read down each column before moving to the next one. However, one interesting artifact is that at the bottom of the first column it extracts the numbers from the section headings which go across the page horizontally. This might make it difficult for the RAG system to refer to a specific section number.

In [ ]:
for page_num, page in enumerate(document2):
    print(f"=== Page {page_num + 1} ===")
    print(page.page_content)
    print()

The second document looks good. The only mistake I noticed is that the loader added a space in the middle of the word 'quantitative'. I don't think there's a fix for this, but I don't think it's too much of a problem since I imagine an LLM will understand that the space isn't meant to be there.

### Cleaning page content

In document 2, there's a lot of whitespace e.g between section headings and the text in the given section. We could remove this whitespace so that the section heading and text are more likely to appear in the same chunk, which helps preserve context and also reduces token usage.

Let's write a function to clean the page content:
- Collapse whitespace, i.e multiple spaces or tabs back-to-back
- Wherever 3 or more new lines occur consecutively, replace them with 2 new lines

In particular, the gaps between paragraphs look like `\n \n \n` (i.e there is a space between new lines), so we'll need to account for that in the regex.

The new line processing is informed by the fact that LangChain's `RecursiveCharacterTextSplitter` -- which we'll use for chunking -- looks for `\n\n` as the plain-text representation of a paragraph break.

In [ ]:
def clean_text(text):
    text = re.sub(r"[ \t]+", " ", text)  # collapse repeated spaces/tabs
    text = re.sub(r"[ \t]+\n", "\n", text)  # strip trailing space before a newline
    text = re.sub(
        r"\n{3,}", "\n\n", text
    )  # collapse blank-line runs to one paragraph break
    return text.strip()


assert clean_text(" Hello   World ") == "Hello World"
assert clean_text("Hello\n\n\nWorld") == "Hello\n\nWorld"
assert clean_text("Hello\n \n \nWorld") == "Hello\n\nWorld"

There are no line breaks in document 1, so let's test this cleaning function on document 2.

In [ ]:
loader = PyPDFLoader(doc2_path)
documents = loader.load()

doc2_page_content = documents[1].page_content
print(doc2_page_content)

In [ ]:
cleaned_content = clean_text(doc2_page_content)
print(cleaned_content)

As expected the multiple-line-breaks are reduced to single line breaks.

### Explore the extracted metadata

In [ ]:
for page_num, page in enumerate(documents):
    print(f"=== Page {page_num + 1} ===")
    print(page.metadata)
    print()

Looking at the metadata, there are a few things that would be useful to persist to the chunking step:
- `source`: currently this gives the whole file path, but we're just interested in the file name
- `page`: this will enable citations referencing the specific page number on which the chunk appears in the original document

We're not interested in metadata such as `producer`, `creator` and `moddate` as they don't relate to the document's content.

## Directory loader

In [ ]:
directory_path = "/Users/ashapatel/Documents/projects/rag_cc/documents/"

loader = DirectoryLoader(
    directory_path, glob="**/*.pdf", loader_cls=PyPDFLoader, show_progress=True
)
documents = loader.load()

print(f"Number of Documents: {len(documents)}")


for idx, document in enumerate(documents, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

This loads all documents in a given directory. This could be useful for my pipeline, but to start with I think I'll specify each document path separately.

## Classify page elements

This loader is suited to PDFs with complex layouts, or scanned documents which require OCR to extract the text. It classifies page elements such as titles, tables, paragraphs etc.

In [ ]:
loader1 = UnstructuredPDFLoader(doc1_path)
documents1 = loader1.load()

print(f"Number of Documents: {len(documents1)}")

for idx, document in enumerate(documents1, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

For document 1 this loader seems to extract an inconsistent line length. It also puts multiple bullet points onto the same line, which seems strange.

In [ ]:
loader2 = UnstructuredPDFLoader(doc2_path)
documents2 = loader2.load()

print(f"Number of Documents: {len(documents2)}")

for idx, document in enumerate(documents2, start=1):
    print(f"\nDocument {idx}")
    print("Content:\n", document.page_content)
    print("Metadata:\n", document.metadata)

For document 2, each section number is put on a separate line from the section heading, which didn't happen with PyPDFLoader.

Even if this loader did work better than simpler ones, we'd need to consider the extra processing time it takes.

## Parsing: Future Work

#### Scaling to more documents

I am starting with a small dataset that I can manually inspect, but this wouldn't scale if I was working with 1000s of documents. In that case, the quality of the parse becomes more important in order to reduce the need for format-specific cleaning.


#### Golden test set

With a larger dataset I'd label a small set of samples with diverse layouts. Each time I make a change to the parsing/cleaning pipeline I could look at how this affects the samples in the golden data set.


#### Detecting anomalies

I could select features such as chunk length, lines with single characters etc and calculate the distribution of these features across all documents. I could then manually investigate anomalies.

The important thing to come back to is the effect of parsing & cleaning on retrieval. I'd have a golden dataset of question-answer pairs with associated relevant chunks in order to evaluate retrieval, and I could measure any changes that result from changes to parsing / cleaning.


## Fixed-size chunking

My initial concern from looking at how formats differ between leaflets in my dataset is that the amount of text in each section can vary a lot. For example, the sections in document 1 contain a lot of text, while in document 2 they're much shorter, often a single line of text per section. 

If I split the documents into fixed-size chunks, then some chunks will contain a lot more sections than others. Since sections in theory represent distinct ideas / pieces of information, this means chunks could end up containing too many distinct ideas. If one of many small sections within a chunk contained information relevant to a query, then the embedding for the whole chunk might not represent that information very well and it might not be retrieved.

Let's have a look at the two example documents I chose to demonstrate this issue:

In [ ]:
clean_text_1 = parse_document(doc1_path)

doc_1_page_1_clean_text = clean_text_1[0].page_content
print(doc_1_page_1_clean_text)

In [ ]:
clean_text_2 = parse_document(doc2_path)

doc_2_page_1_clean_text = clean_text_2[0].page_content
print(doc_2_page_1_clean_text)

Let's apply a fixed-size splitter, e.g `RecursiveCharacterTextSplitter`, and have a look at the resulting chunks.

Here's how I'd go about choosing `chunk_size` & `chunk_overlap` for this splitter:

- In order to set `chunk_size`, we need to take into account the maximum sequence length of the embedding model that we'll be using. We can revisit this once we've chosen an embedding model, but for now let's work off of the assumption that 256 is a safe number of tokens for most sentence-transformer models. Assuming ~4 characters per token, this gives a chunk size of ~1000.
    - Note that in our production-ready code, parameters such as `chunk_size` will be set in a config file, not hardcoded.
- It is generally recommended that the overlap between chunks should be set to 10-20% of the chunk size. We'll take 15% of 1000 which is 150. It's good to remember that the purpose of this overlap is to maintain context between one chunk and the next. Without overlap, an idea could get split between two chunks. However, setting the overlap too high wastes storage space in the vector store and wastes compute during retrieval.
- `RecursiveCharacterTextSplitter` also has a `separators` parameter which defaults to `["\n\n", "\n", " ", ""]`. This means it first tries to split on paragraphs, then new lines, then words, then characters.

In [ ]:
doc_1_page_1_chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150
).split_text(doc_1_page_1_clean_text)

In [ ]:
doc_2_page_1_chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150
).split_text(doc_2_page_1_clean_text)

In [ ]:
for chunk in doc_1_page_1_chunks:
    print(chunk)
    print("====================")

In [ ]:
for chunk in doc_2_page_1_chunks:
    print(chunk)
    print("====================")

For document 1 we can see that a single section is sometimes split over multiple chunks, whereas for document 2 chunks contain multiple sections.

If we wanted 1 section per chunk, how could we do that?

In document 1 the main section headings are numbered 1-6, but within these sections there are sometimes sub-sections which are not numbered but which appeared in bold text in the original document.

In document 2 each section is numbered 1-10 and sub-sections are numbered too (e.g 4.1, 4.2).

We can't just define a regex to detect sections in both documents. I think we need to revisit the parsing step and use a loader that detects section headers with various formats, whether numbered or bold. We previously tried `UnstructuredPDFLoader`, however we didn't explore all of its modes, and it sounds like they might be well suited to our issue.

## Layout-based parsing

### UnstructuredPDFLoader

Let's try `UnstructuredPDFLoader` again, but this time with `mode="elements"` instead of the default `mode="single"`. In `"single"` mode the loader still runs its internal layout analysis, but then joins everything back into one block of text, which is why we saw section numbers get separated from headings and bullet points get merged together. In `"elements"` mode it instead returns one `Document` per detected element (title, narrative text, list item, ...), with the element type in `metadata["category"]`. That's what we need in order to find section boundaries automatically.

There are two `strategy` options worth comparing:
- `"fast"`: text-only heuristics, no extra system dependencies.
- `"hi_res"`: uses a layout-detection model, which should be better at catching headings that aren't numbered (like doc 1's bold sub-headings). This needs `poppler` and `tesseract` installed on the system (e.g. `brew install poppler tesseract` on macOS).

In [ ]:
def pdf_parser(path: str, strategy: str) -> list[Document]:
    loader = UnstructuredPDFLoader(path, mode="elements", strategy=strategy)
    elements = loader.load()
    print(f"{len(elements)} elements\n")
    for element in elements:
        category = element.metadata.get("category")
        print(f"{category:<20} {element.page_content}")
    return elements

#### `strategy="fast"`

In [ ]:
doc1_elements_fast = pdf_parser(doc1_path, strategy="fast")

A few observations:
- The main title (large, centered text at the top of the document) is labelled as `NarrativeText` when I'd expect it to be labelled as `Title`
- Some bullet points are extracted as part of an element labelled as `NarrativeText`
- The first heading that appears as white text in a black box gets labelled as a title, but the next section title that appears in this format gets labelled as `NarrativeText`

In [ ]:
doc2_elements_fast = pdf_parser(doc2_path, strategy="fast")

A few observations:
- The first (main) title gets labelled as a title, as expected
- The number 1 indicating the first section heading gets labelled as `UncategorizedText`, while the text on the same line gets labelled as `Title`. This is a consistent pattern with all section headings.

Based on these 2 documents, this loader with these parameters do not fix our issue.

#### `strategy="hi_res"`

This needs `tesseract` on your `PATH` (`poppler` is likely already installed, since `pdf2image` uses `pdftoppm` from it under the hood). If it's missing you'll see a `TesseractNotFoundError` — install it (`brew install tesseract` on macOS) and re-run.

In [ ]:
import os

os.environ["TESSDATA_PREFIX"] = "/opt/homebrew/anaconda3/share/tessdata/"

In [ ]:
doc1_elements_hi_res = pdf_parser(doc1_path, strategy="hi_res")

This seems worse than the `fast` strategy. The loader read across the first two columns instead of straight down the first, and labelled some text as a table. It labelled most of the text as uncategorised.

In [ ]:
doc2_elements_hi_res = pdf_parser(doc2_path, strategy="hi_res")

The output for document 2 looks better. Titles and subtitles get labelled as `Title`, and the text in each section gets labelled as `NarrativeText`.

Overall, it seems like `UnstructuredPDFLoader` doesn't solve my issue. Let's try the `docling` library.

### Docling

Docling is a 'document-conversion toolkit' from IBM. It uses a layout-detection model along with a table-structure model and optional OCR, producing a structured `DoclingDocument` object. Due to popularity it's been integrated with LangChain.

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions,
    AcceleratorDevice,
)
from docling.datamodel.base_models import InputFormat
from docling_core.types.doc.document import DoclingDocument

pipeline_options = PdfPipelineOptions()

# Set to use CPU instead of GPU, otherwise there is an error
pipeline_options.accelerator_options = AcceleratorOptions(device=AcceleratorDevice.CPU)

In [ ]:
def docling_parser(path: str) -> DoclingDocument:
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    doc = converter.convert(path).document
    items = list(doc.iterate_items())
    print(f"{len(items)} items\n")
    for item, level in items:
        label = str(item.label)
        text = getattr(item, "text", "")
        print(f"{label:<20} level={level:<2} {text}")
    return doc

In [ ]:
doc1_parsed = docling_parser(doc1_path)

This looks much better than the previous parsers. Headings are correctly labelled as `section_header`, and within sections even `text` and `list_items`s are mostly distinguished correctly. There are some examples where a bullet point gets merged with the text that follows it instead of getting its own `list_item` label.

The numbers next to the main headings haven't been extracted.

Docling assigns a `level` to each element to indicate 'hierarchical depth'. For example you'd expect a section heading to be the level above the text inside the section. This seems to be generally adhered to but not perfectly.

One thing to note is that it's significantly slower to run.

In [ ]:
doc2_parsed = docling_parser(doc2_path)

Section headers and text seem to be consistently labelled correctly. The numbers next to headings are even extracted, unlike in document 1.

However, everything is given a level 1 label for some reason.

Overall, this is by far the best parser I've tested on these specific documents. Let's proceed to chunking using this parser.

A few things to note:
- It's slow due to the powerful models it uses, however this isn't a problem for this project
- The `level` labels seem unreliable, and this seems to be a fairly common issue. We will just ignore the level labels.

In [ ]:
def reformat_docling_docs(doc: DoclingDocument) -> list[dict]:
    """
    Reformat a docling document into a list of dictionaries, where each dictionary contains a section heading and its corresponding text.
    """

    sections = []
    current = None
    for item, level in doc.iterate_items():
        label = item.label
        text = getattr(item, "text", "")
        if label in (DocItemLabel.TITLE, DocItemLabel.SECTION_HEADER):
            current = {"heading": text, "text": ""}
            sections.append(current)
        else:
            if current is None:
                current = {"heading": None, "text": ""}
                sections.append(current)
            current["text"] += text + "\n"
    return sections

In [ ]:
doc1_sections = reformat_docling_docs(doc1_parsed)
doc1_sections

In [ ]:
doc2_sections = reformat_docling_docs(doc2_parsed)
doc2_sections

## Chunking per-section

We've already discussed why using fixed sized chunks isn't well suited to these documents, and why using one section per chunk might be better. There are pros and cons to this, but we'll pursue this for now and see how the RAG system performs.

In the parsing step we add the following metadata per section, where we identify a new section each time docling assigns a new label e.g `title` or `section header`:
- source: document name, e.g `paracetamol_PIL.pdf`
- medicine_name: extracted from the document name by removing `_PIL.pdf`
- section["heading"]: heading text for the current section
- section["page"]: page number from the original document
- section_index: position of section within the document

To get from document sections to chunks, we want to do the following:
- set a chunk size, and split up sections with page content larger than the chunk size (using `RecursiveCharacterTextSplitter`)
- add a `chunk index` to the metadata, indicating the position of the chunk within its section. This is often 0 since most sections become a single chunk
- add the medicine name & section heading to the chunk's page content, so that they form part of the text that gets embedded (in addition to the section text itself)

Each chunk already contains the metadata added in the parsing step.

In [ ]:
def split_docs_into_chunks(documents: list, chunk_size=1000, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    chunks = []
    for document in documents:
        sub_documents = (
            [document]
            if len(document.page_content) <= chunk_size
            else splitter.split_documents([document])
        )
        for chunk_index, chunk in enumerate(sub_documents):
            chunk.metadata["chunk_index"] = chunk_index
            prefix_lines = []
            if medicine := chunk.metadata.get("medicine_name"):
                prefix_lines.append(f"Medicine: {medicine}")
            if heading := chunk.metadata.get("heading"):
                prefix_lines.append(f"Section: {heading}")
            if prefix_lines:
                chunk.page_content = (
                    "\n".join(prefix_lines) + "\n\n" + chunk.page_content
                )
            chunks.append(chunk)

    return chunks

In [ ]:
from rag_pipeline.parse_documents import parse_document

In [ ]:
doc1_parsed = parse_document(doc1_path)
doc1_parsed

In [ ]:
doc1_chunks = split_docs_into_chunks(doc1_parsed)
doc1_chunks

Next I'll update `parse_documents.py`, `chunking.py` and `run_indexing.py` and run `run_indexing.py` to build the vector store.

I'll inspect the resulting vector store in `vector_store_exploration.ipynb`.

# Issues

An issue I discovered when building the evaluation pipeline is that the parser sometimes removes spaces between words.

The evaluation pipeline looks for matches between the quotes in the `evidence` of a ground truth sample, and the chunks in the vector store. I found the following mismatch:

- quote from the original document: "if you are breast-feeding"
- quote in the vector store: "if you arebreast-feeding"

The simplest fix would be to adjust the `normalise()` function in `retrieval_metrics.py` to remove whitespace from both the original quote and the stored quote.

There are some potential issues that could arise as a result of this:

- Removing spaces from something like `is not` would make it match with `this not`. This is more likely to happen for shorter quotes, so I should make sure that my quotes are not too short
- The joined-up words would still exist in the vector store and this might affect the emeddings and therefore retrieval quality.

Since this issue only happens in one document, I'll make the change to `normalise()` for now. In the future, I could spend more time experimenting with parsers to make sure that this issue didn't happen in the first place.